In [91]:
import pandas as pd
import numpy as np

In [92]:
fred = pd.read_csv('fred_unemployment_raw.csv')
elections = pd.read_csv('election_results_1976_2024.csv')

In [93]:
fred.head()

,date,state,series_id,unemployment_rate,realtime_start,realtime_end
0,1976-01-01,Alabama,ALUR,6.7,2026-03-27,2026-03-27
1,1976-02-01,Alabama,ALUR,6.6,2026-03-27,2026-03-27
2,1976-03-01,Alabama,ALUR,6.600,2026-03-27,2026-03-27
3,1976-04-01,Alabama,ALUR,6.500,2026-03-27,2026-03-27
4,1976-05-01,Alabama,ALUR,6.400,2026-03-27,2026-03-27


In [94]:
elections.head()

,Year,State,Region,Democratic %,Republican %,P.S.,P.S. Score,National Winner,State Winner
0,1976,Alabama,South,55.7%,42.6%,D+,11.23,Carter (D),Democrat
1,1980,Alabama,South,47.4%,48.8%,D+,9.26,Reagan (R),Democrat
2,1984,Alabama,South,38.3%,60.5%,R+,-4.19,Reagan (R),Republican
3,1988,Alabama,South,39.9%,59.2%,R+,-11.69,Bush H.W. (R),Republican
4,1992,Alabama,South,40.9%,47.6%,R+,-14.55,Clinton (D),Republican


In [95]:
elections["Democratic %"] = elections["Democratic %"].str.replace("%", "").astype(float)
elections["Republican %"] = elections["Republican %"].str.replace("%", "").astype(float)

In [96]:
elections["State"] = elections["State"].str.strip().str.title()
fred["state"] = fred["state"].str.strip().str.title()

In [97]:
elections = elections[elections["State"] != "Washington D.C."]

In [98]:
fred["date"] = pd.to_datetime(fred["date"])
fred["Year"] = fred["date"].dt.year
fred["Month"] = fred["date"].dt.month

In [99]:
fred_nov = fred[fred["Month"] == 11].copy()

In [100]:
fred_nov = fred_nov[["Year", "state", "unemployment_rate"]]
fred_nov = fred_nov.rename(columns={"state": "State"})

In [101]:
fused = pd.merge(
    elections,
    fred_nov,
    on=["Year", "State"],
    how="left"
)

In [102]:
fused["unemployment_rate"] = pd.to_numeric(fused["unemployment_rate"], errors="coerce")

In [103]:
fused = fused.sort_values(["State", "Year"])

fused["unemployment_change"] = (
    fused.groupby("State")["unemployment_rate"].diff()
)

In [104]:
fused = fused.rename(columns={
    "unemployment_rate": "ElectionYear_Unemployment",
    "unemployment_change": "Change_Since_Last_Election"
})

In [105]:
fused.head()

,Year,State,Region,Democratic %,Republican %,P.S.,P.S. Score,National Winner,State Winner,ElectionYear_Unemployment,Change_Since_Last_Election
0,1976,Alabama,South,55.7,42.6,D+,11.23,Carter (D),Democrat,7.0,NaN
1,1980,Alabama,South,47.4,48.8,D+,9.26,Reagan (R),Democrat,9.4,2.4
2,1984,Alabama,South,38.3,60.5,R+,-4.19,Reagan (R),Republican,10.5,1.1
3,1988,Alabama,South,39.9,59.2,R+,-11.69,Bush H.W. (R),Republican,7.3,-3.2
4,1992,Alabama,South,40.9,47.6,R+,-14.55,Clinton (D),Republican,7.5,0.2


In [106]:
fused.to_csv("fused.csv")

In [107]:
import hashlib
with open("fused.csv", "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
with open("fused.csv.sha", 'w') as f:
    f.write(sha256)

650